In [2]:
from pyspark.sql import Row
from datetime import datetime

# Drop existing table (optional but recommended if schema changed)
spark.sql("DROP TABLE IF EXISTS pipeline_watermarks")

# Create the watermark table
watermark_df = spark.createDataFrame([
    Row(
        table_name="silver_orders",
        last_loaded_ts=datetime(2000, 1, 1, 0, 0, 0)
    )
])

# Save as Delta table
watermark_df.write \
    .format("delta") \
    .saveAsTable("pipeline_watermarks")

print("Watermark table created successfully.")

# Verify the contents
spark.table("pipeline_watermarks").show(truncate=False)

# Verify the schema
spark.table("pipeline_watermarks").printSchema()

StatementMeta(, 75c124ab-00be-4a62-9769-a84e1f74d667, 4, Finished, Available, Finished, False)

Watermark table created successfully.
+-------------+-------------------+
|table_name   |last_loaded_ts     |
+-------------+-------------------+
|silver_orders|2000-01-01 00:00:00|
+-------------+-------------------+

root
 |-- table_name: string (nullable = true)
 |-- last_loaded_ts: timestamp (nullable = true)



In [4]:
from pyspark.sql.functions import col, lit
from delta.tables import DeltaTable
from datetime import datetime

# Step 1: Read the watermark
wm_rows = (
    spark.table("pipeline_watermarks")
    .filter(col("table_name") == "silver_orders")
    .collect()
)

if not wm_rows:
    raise ValueError(
        "No watermark found for 'silver_orders'. "
        "Please recreate the pipeline_watermarks table."
    )

last_ts = wm_rows[0]["last_loaded_ts"]

print(f"Loading rows with processed_timestamp > {last_ts}")

# Step 2: Read only new rows
df_new = (
    spark.table("silver_orders")
    .filter(col("processed_timestamp") > lit(last_ts))
)

new_row_count = df_new.count()

print(f"New rows to process: {new_row_count}")

# Step 3: Exit if nothing new
if new_row_count == 0:
    print("No new data. Pipeline complete.")

else:

    # Load dimensions
    dim_cust = (
        spark.table("dim_customer")
        .filter(col("is_current") == True)
        .dropDuplicates(["customer_id"])
    )

    dim_prod = (
        spark.table("dim_product")
        .dropDuplicates(["product_id"])
    )

    dim_date = (
        spark.table("dim_date")
        .dropDuplicates(["order_date"])
    )

    # Build fact rows
    new_facts = (
        df_new
        .join(
            dim_cust.select("customer_id", "customer_key"),
            on="customer_id",
            how="left"
        )
        .join(
            dim_prod.select("product_id", "product_key"),
            on="product_id",
            how="left"
        )
        .join(
            dim_date.select("order_date", "date_key"),
            on="order_date",
            how="left"
        )
        .select(
            "order_id",
            "customer_key",
            "product_key",
            "date_key",
            "region",
            "sales",
            "quantity",
            "discount",
            "profit"
        )
        .dropDuplicates(["order_id"])
    )

    # Merge into fact table
    dt = DeltaTable.forName(spark, "fact_orders")

    (
        dt.alias("target")
        .merge(
            new_facts.alias("source"),
            "target.order_id = source.order_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(f"fact_orders updated: {new_facts.count()} rows merged.")

    # Step 4: Update watermark
    DeltaTable.forName(spark, "pipeline_watermarks").update(
        condition=col("table_name") == "silver_orders",
        set={
            "last_loaded_ts": lit(datetime.now())
        }
    )

    print("Watermark updated successfully.")

StatementMeta(, 75c124ab-00be-4a62-9769-a84e1f74d667, 6, Finished, Available, Finished, False)

Loading rows with processed_timestamp > 2000-01-01 00:00:00
New rows to process: 10194
fact_orders updated: 5111 rows merged.
Watermark updated successfully.


In [5]:
from pyspark.sql.functions import col, count

new_facts.groupBy("order_id") \
    .agg(count("*").alias("cnt")) \
    .filter(col("cnt") > 1) \
    .show()

StatementMeta(, 75c124ab-00be-4a62-9769-a84e1f74d667, 7, Finished, Available, Finished, False)

+--------+---+
|order_id|cnt|
+--------+---+
+--------+---+



In [4]:
new_facts = new_facts.dropDuplicates(["order_id"])

StatementMeta(, 762abf91-103e-46cb-9fe4-28cc36acc1e1, 6, Finished, Available, Finished, False)

In [5]:
new_facts.groupBy("order_id").count().filter("count > 1").show()

StatementMeta(, 762abf91-103e-46cb-9fe4-28cc36acc1e1, 7, Finished, Available, Finished, False)

+--------+-----+
|order_id|count|
+--------+-----+
+--------+-----+



In [2]:
from pyspark.sql import Row
from datetime import datetime

# Drop the existing table if it exists
spark.sql("DROP TABLE IF EXISTS pipeline_watermarks")

# Create the watermark DataFrame
watermark_df = spark.createDataFrame([
    Row(
        table_name="silver_orders",
        last_loaded_ts=datetime(2000, 1, 1, 0, 0, 0)
    )
])

# Save as a new Delta table
watermark_df.write \
    .format("delta") \
    .saveAsTable("pipeline_watermarks")

print("Watermark table recreated with timestamp column.")

# Verify the data
spark.table("pipeline_watermarks").show(truncate=False)

# Verify the schema
spark.table("pipeline_watermarks").printSchema()

StatementMeta(, ccd57226-6f1f-4b54-89b7-90a021dae88e, 4, Finished, Available, Finished, False)

Watermark table recreated with timestamp column.
+-------------+-------------------+
|table_name   |last_loaded_ts     |
+-------------+-------------------+
|silver_orders|2000-01-01 00:00:00|
+-------------+-------------------+

root
 |-- table_name: string (nullable = true)
 |-- last_loaded_ts: timestamp (nullable = true)

